Let's use MCP in OpenAI Agents SDK

Create a Client

Have it spawn a server

Collect the tools that the server can use

In [1]:
import asyncio
from agents.mcp import MCPServerStdio

fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params) as server:
    tools = await server.list_tools()
    #print(tools)

for tool in tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")

fetch: Fetches a URL from the internet and optionally extracts its contents as markdown.  Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.


Exploring Node-Based Servers & Tool Integrations

The next MCP tool uses node (the Javascript Server), and it needs you to have the command 'npx' installed on your computer.

In [2]:
puppeteer_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-puppeteer"]
}

async with MCPServerStdio(params=puppeteer_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

    
for tool in tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")

puppeteer_navigate: Navigate to a URL
puppeteer_screenshot: Take a screenshot of the current page or a specific element
puppeteer_click: Click an element on the page
puppeteer_fill: Fill out an input field
puppeteer_select: Select an element on the page with Select tag
puppeteer_hover: Hover an element on the page
puppeteer_evaluate: Execute JavaScript in the browser console


In [3]:
playwright_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest"]
}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")

browser_close: Close the page
browser_resize: Resize the browser window
browser_console_messages: Returns all console messages
browser_handle_dialog: Handle a dialog
browser_emulate_media: Emulate CSS media features for the page, for example switch between the light and dark color scheme. Omitted parameters are left unchanged; null clears an override.
browser_evaluate: Evaluate JavaScript expression on page or element
browser_file_upload: Upload one or multiple files
browser_drop: Drop files or MIME-typed data onto an element, as if dragged from outside the page. At least one of "paths" or "data" must be provided.
browser_find: Search the accessibility snapshot of the current page for text or a regular expression. Returns matching snapshot nodes with a few lines of surrounding context (like search snippets), each shown under its path from the root of the tree, which is cheaper than capturing the whole snapshot when you only need to locate an element and its ref.
browser_fill_form: Fill

In [4]:
fileSystem_path = "/Users/anshumanipathak/AgenticAI_New/Resources"

files_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", fileSystem_path]
}

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")

read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
read_text_file: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
read_media_file: Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.
read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files.

Explore Integrations of Multiple MCP Servers

In [9]:
import asyncio
from agents.mcp import MCPServerStdio
from openai import OpenAI
from openai import AsyncOpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents import OpenAIChatCompletionsModel, set_tracing_disabled
load_dotenv(override=True)
set_tracing_disabled(disabled=True)
import os

fileSystem_path = "/Users/anshumanipathak/AgenticAI_New/Resources"

files_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", fileSystem_path]
}

playwright_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest"]
}

# Clear browsing instructions for the agent
agent_instructions = """
You use the internet efficiently to complete your objectives.  
You are fully capable of navigating websites on your own — accepting cookies, dismissing pop-ups, or selecting “not now” when necessary to access the required information.  
If one source does not provide useful results, move on to another.  
Stay determined and keep exploring different pages and resources until the task is successfully completed.
"""

#Connect to deepseek
deepseek_client = AsyncOpenAI(
    api_key=os.environ.get('API_TOKEN'),
    base_url="https://api.deepseek.com")

deepseek_model = OpenAIChatCompletionsModel(
    model="deepseek-flash",
    openai_client=deepseek_client
)


#Create the mcp server and run the agent
async with MCPServerStdio(params=files_params, cache_tools_list=True, client_session_timeout_seconds=30) as file_server:
    async with MCPServerStdio(params=playwright_params, cache_tools_list=True, client_session_timeout_seconds=30) as web_server:
        agent = Agent(
            name="Internet Researcher",
            instructions=agent_instructions,
            model=deepseek_model,
            mcp_servers=[file_server, web_server]
        )

        task = "Find the best authentic recipe for Idle Dosha and write it to mind_blowing_idle.md in Markdown format."

        with trace("mind_blowing_idle_recipe_task"):
            result = await Runner.run(agent, task)
            print("\n✅ Final Output:\n")
            print(result.final_output)


MaxTurnsExceeded: Max turns (10) exceeded